# Laborator 4 – Rețele Markov

In [ ]:
import itertools
import numpy as np
import pyagrum as gum
import pyagrum.lib.notebook as gnb

beta = 1.0
vars_ = ["A1", "A2", "A3", "A4", "A5"]
edges = [("A1","A2"), ("A1","A3"), ("A2","A4"), ("A2","A5"), ("A3","A4"), ("A4","A5")]

## a) Graful și clicile maximale

In [ ]:
cliques = [("A1","A2"), ("A1","A3"), ("A3","A4"), ("A2","A4","A5")]

E = {frozenset(e) for e in edges}
assert all(frozenset(p) in E for c in cliques for p in itertools.combinations(c, 2))
assert E == {frozenset(p) for c in cliques for p in itertools.combinations(c, 2)}

mn = gum.MarkovRandomField()
for v in vars_:
    mn.add(gum.LabelizedVariable(v, v, 2))
for c in cliques:
    mn.addFactor(list(c))

print("Clici maximale:", cliques)
gnb.showMRF(mn)

## b) Probabilitatea comună și stările de probabilitate maximă

In [ ]:
def phi(vals):
    return np.exp(beta * sum(x*y for x, y in itertools.combinations(vals, 2)))

def unnorm(s):
    a = dict(zip(vars_, s))
    return np.prod([phi([a[v] for v in c]) for c in cliques])

states = list(itertools.product([0, 1], repeat=5))
Z = sum(unnorm(s) for s in states)
P = {s: unnorm(s) / Z for s in states}

for s, p in P.items():
    print(s, round(p, 5))

print("Z =", Z, "| sumă P =", sum(P.values()))
pmax = max(P.values())
print("Stări de probabilitate maximă:", [s for s, p in P.items() if np.isclose(p, pmax)], pmax)

Pentru β > 0, exponentul `β·(a1a2 + a1a3 + a3a4 + a2a4 + a2a5 + a4a5)` este maxim pentru `(1,1,1,1,1)`, adică `6β`.
Deci `P = e^{6β} / Z`.

## c) Modelul Ising

Se folosesc doar potențiale pe perechi de vecini (muchii), care favorizează valori egale:

- variabile {0,1}: `φ(ai, aj) = e^β` dacă `ai = aj`, altfel `1` (β > 0)
- spini {−1,+1}: `φ(ai, aj) = e^{β·ai·aj}` (β > 0)

Clica {A2,A4,A5} se descompune în `φ(A2,A4)·φ(A2,A5)·φ(A4,A5)`.

In [ ]:
def phi_ising(ai, aj):
    return np.exp(beta) if ai == aj else 1.0

def unnorm_ising(s):
    a = dict(zip(vars_, s))
    return np.prod([phi_ising(a[x], a[y]) for x, y in edges])

Z_i = sum(unnorm_ising(s) for s in states)
P_i = {s: unnorm_ising(s) / Z_i for s in states}
pm = max(P_i.values())
print("Stări maxime (Ising):", [s for s, p in P_i.items() if np.isclose(p, pm)], pm)